# Investment Risk Prediction: Random Forest vs XGBoost

Google Colab notebook for RELIANCE.NS, TCS.NS, INFY.NS and HDFCBANK.NS.

**Important:** 80%+ accuracy cannot be guaranteed honestly for stock-risk prediction. This notebook uses a chronological holdout, training-period-only risk thresholds, and a 20-day gap to reduce leakage. It compares Random Forest and XGBoost and selects the model with the better held-out weighted F1 (tie-break: accuracy).

## 1. Install libraries

In [ ]:
%pip -q install yfinance xgboost joblib

In [ ]:
import json, shutil, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, yfinance as yf, joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
SEED=42


## 2. Download daily stock data

In [ ]:
TICKERS = ["RELIANCE.NS", "TCS.NS", "INFY.NS", "HDFCBANK.NS"]
start = "2016-01-01"
end = (pd.Timestamp.today().normalize() + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
parts=[]
for ticker in TICKERS:
    print("Downloading", ticker)
    d=yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False, multi_level_index=False)
    if d is None or d.empty:
        print("WARNING: no data for", ticker); continue
    d=d.copy()
    d.columns=[str(c[0] if isinstance(c,tuple) else c) for c in d.columns]
    d=d.reset_index()
    d=d.rename(columns={d.columns[0]:"Date"})
    d["Stock"]=ticker
    parts.append(d)
if not parts: raise RuntimeError("No stock data downloaded. Check internet and rerun.")
prices=pd.concat(parts,ignore_index=True)
prices["Date"]=pd.to_datetime(prices["Date"],errors="coerce")
prices=prices.dropna(subset=["Date","Close"]).drop_duplicates(["Date","Stock"]).sort_values(["Stock","Date"])
print("Rows:",len(prices),"Date range:",prices.Date.min().date(),"to",prices.Date.max().date())
display(prices.groupby("Stock").agg(first_date=("Date","min"),last_date=("Date","max"),rows=("Close","count")))


## 3. Plot closing prices

In [ ]:
plt.figure(figsize=(14,5))
for s,g in prices.groupby("Stock"):
    plt.plot(g.Date,g.Close,label=s,linewidth=1)
plt.title("Historical adjusted closing prices"); plt.xlabel("Date"); plt.ylabel("Price (INR)")
plt.legend(); plt.grid(alpha=.25); plt.tight_layout(); plt.show()


## 4. Build past-only features and future volatility target

In [ ]:
def make_features(g):
    g=g.sort_values("Date").copy()
    c=g["Close"].astype(float); r=c.pct_change()
    g["Daily_Return"]=r
    g["Return_5D"]=c.pct_change(5)
    g["Volatility_20"]=r.rolling(20).std()
    delta=c.diff()
    gain=delta.clip(lower=0).rolling(14).mean()
    loss=(-delta.clip(upper=0)).rolling(14).mean()
    rs=gain/loss.replace(0,np.nan)
    g["RSI_14"]=100-100/(1+rs)
    g["Price_MA5_Ratio"]=c/c.rolling(5).mean()
    g["Price_MA20_Ratio"]=c/c.rolling(20).mean()
    g["Return_20D"]=c.pct_change(20)
    g["Volatility_5"]=r.rolling(5).std()
    # At date t, this uses returns from t+1 through t+20.
    g["Future_Volatility_20"]=r.shift(-1).iloc[::-1].rolling(20,min_periods=20).std().iloc[::-1]
    return g
featured=pd.concat([make_features(g) for _,g in prices.groupby("Stock")],ignore_index=True)
FEATURES=["Daily_Return","Return_5D","Volatility_20","RSI_14","Price_MA5_Ratio","Price_MA20_Ratio","Return_20D","Volatility_5"]
featured=featured.replace([np.inf,-np.inf],np.nan)
model_df=featured.dropna(subset=FEATURES+["Future_Volatility_20"]).sort_values(["Date","Stock"]).reset_index(drop=True)
print("Usable rows:",len(model_df)); display(model_df.head())


## 5. Chronological split and risk classes (thresholds learned on training only)

In [ ]:
dates=np.array(sorted(model_df.Date.unique()))
split_pos=int(len(dates)*0.80)
if len(dates)<100 or split_pos<=20: raise ValueError("Not enough dates to split.")
test_start=pd.Timestamp(dates[split_pos])
gap_start=pd.Timestamp(dates[max(0,split_pos-20)])
train=model_df[model_df.Date<gap_start].copy()
test=model_df[model_df.Date>=test_start].copy()
q_low,q_high=train["Future_Volatility_20"].quantile([1/3,2/3]).tolist()
def risk_label(v): return np.select([v<=q_low,v<=q_high],[0,1],default=2).astype(int)
train["Risk_Label"]=risk_label(train["Future_Volatility_20"])
test["Risk_Label"]=risk_label(test["Future_Volatility_20"])
X_train=train[FEATURES].astype(float); y_train=train.Risk_Label.astype(int)
X_test=test[FEATURES].astype(float); y_test=test.Risk_Label.astype(int)
print("Train rows:",len(train),"Test rows:",len(test),"20-date gap starts:",gap_start.date(),"Test begins:",test_start.date())
print("Train-only volatility cutoffs:",q_low,q_high)
print("Train class counts:",y_train.value_counts().sort_index().to_dict())
print("Test class counts:",y_test.value_counts().sort_index().to_dict())


## 6. Train Random Forest and XGBoost

In [ ]:
weights=compute_sample_weight(class_weight="balanced",y=y_train)
rf_model=RandomForestClassifier(n_estimators=400,max_depth=12,min_samples_leaf=4,max_features="sqrt",class_weight="balanced_subsample",random_state=SEED,n_jobs=-1)
rf_model.fit(X_train,y_train)
xgb_model=XGBClassifier(n_estimators=400,max_depth=4,learning_rate=.04,subsample=.85,colsample_bytree=.85,reg_lambda=2,objective="multi:softprob",num_class=3,eval_metric="mlogloss",random_state=SEED,n_jobs=-1,tree_method="hist")
xgb_model.fit(X_train,y_train,sample_weight=weights)
print("Both models trained.")


## 7. Compare test performance and select the better model

In [ ]:
def metrics(name,model):
    p=model.predict(X_test)
    return {"Model":name,"Accuracy":accuracy_score(y_test,p),"Balanced Accuracy":balanced_accuracy_score(y_test,p),
            "Weighted Precision":precision_score(y_test,p,average="weighted",zero_division=0),
            "Weighted Recall":recall_score(y_test,p,average="weighted",zero_division=0),
            "Weighted F1":f1_score(y_test,p,average="weighted",zero_division=0),
            "Macro F1":f1_score(y_test,p,average="macro",zero_division=0)}
results=pd.DataFrame([metrics("Random Forest",rf_model),metrics("XGBoost",xgb_model)]).sort_values(["Weighted F1","Accuracy"],ascending=False).reset_index(drop=True)
display(results.style.format({c:"{:.2%}" for c in results.columns if c!="Model"}))
best_name=results.loc[0,"Model"]
best_model=rf_model if best_name=="Random Forest" else xgb_model
print("Selected model:",best_name)
print("Held-out accuracy:",f"{results.loc[0,'Accuracy']:.2%}")
if results.loc[0,"Accuracy"]<.80:
    print("Accuracy is below 80%. This is a valid result; do not manipulate the test split or labels to inflate it.")


In [ ]:
class_names=["Low Risk","Medium Risk","High Risk"]
for name,model in [("Random Forest",rf_model),("XGBoost",xgb_model)]:
    pred=model.predict(X_test)
    print("\\n",name)
    print(classification_report(y_test,pred,labels=[0,1,2],target_names=class_names,zero_division=0))
    sns.heatmap(confusion_matrix(y_test,pred,labels=[0,1,2]),annot=True,fmt="d",cmap="Blues",xticklabels=class_names,yticklabels=class_names)
    plt.title(name+" Confusion Matrix"); plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.tight_layout(); plt.show()


## 8. Feature importance

In [ ]:
importance=pd.DataFrame({"Feature":FEATURES,"Importance":best_model.feature_importances_}).sort_values("Importance",ascending=False)
display(importance)
sns.barplot(data=importance,x="Importance",y="Feature")
plt.title("Feature importance - "+best_name); plt.tight_layout(); plt.show()


## 9. Save trained models and export data for the future Streamlit app

In [ ]:
EXPORT=Path("/content/InvestRisk_Export")
(EXPORT/"artifacts").mkdir(parents=True,exist_ok=True)
(EXPORT/"data").mkdir(parents=True,exist_ok=True)
joblib.dump(rf_model,EXPORT/"artifacts"/"random_forest.joblib")
joblib.dump(xgb_model,EXPORT/"artifacts"/"xgboost.joblib")
(EXPORT/"artifacts"/"feature_columns.json").write_text(json.dumps(FEATURES,indent=2),encoding="utf-8")
(EXPORT/"artifacts"/"risk_thresholds.json").write_text(json.dumps({
    "low_medium_threshold":float(q_low),"medium_high_threshold":float(q_high),
    "class_mapping":{"0":"Low Risk","1":"Medium Risk","2":"High Risk"},"selected_model":best_name,
    "target":"standard deviation of next 20 daily returns"},indent=2),encoding="utf-8")
out=model_df.copy()
out["Risk_Label"]=risk_label(out["Future_Volatility_20"])
out[["Date","Stock","Close","Risk_Label","Future_Volatility_20"]+FEATURES].to_csv(EXPORT/"data"/"model_data.csv",index=False)
results.to_csv(EXPORT/"data"/"model_metrics.csv",index=False)
zip_path=shutil.make_archive("/content/InvestRisk_Export","zip",root_dir=EXPORT)
print("Created:",zip_path,"| Selected:",best_name,"| Accuracy:",f"{results.loc[0,'Accuracy']:.2%}")
from google.colab import files
files.download(zip_path)


## Notes
- Accuracy above 80% is a target, not a guarantee.
- The selected model is based on held-out weighted F1, with accuracy as a tie-breaker, rather than choosing whichever has a flattering result.
- Stock volatility is noisy and regime-dependent. The risk categories are relative to this training dataset.
- The models are for academic use, not financial advice.
- yfinance data can change over time; document the date you run the notebook.